<a href="https://colab.research.google.com/github/ngoanlc25ai/AdvanceDataScience/blob/main/Real_ESRGAN_Inference_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-ESRGAN Inference Demo

[![arXiv](https://img.shields.io/badge/arXiv-Paper-<COLOR>.svg)](https://arxiv.org/abs/2107.10833)
[![GitHub Stars](https://img.shields.io/github/stars/xinntao/Real-ESRGAN?style=social)](https://github.com/xinntao/Real-ESRGAN)
[![download](https://img.shields.io/github/downloads/xinntao/Real-ESRGAN/total.svg)](https://github.com/xinntao/Real-ESRGAN/releases)

This is a **Practical Image Restoration Demo** of our paper [''Real-ESRGAN: Training Real-World Blind Super-Resolution with Pure Synthetic Data''](https://arxiv.org/abs/2107.10833).
We extend the powerful ESRGAN to a practical restoration application (namely, Real-ESRGAN), which is trained with pure synthetic data. <br>
The following figure shows some real-life examples.

<img src="https://raw.githubusercontent.com/xinntao/Real-ESRGAN/master/assets/teaser.jpg" width="100%">

We provide a pretrained model (*RealESRGAN_x4plus.pth*) with upsampling X4.<br>
**Note that RealESRGAN may still fail in some cases as the real-world degradations are really too complex.**<br>
Moreover, it **may not** perform well on **human faces, text**, *etc*, which will be optimized later.
<br>

You can also find a **Portable Windows/Linux/MacOS executable files for Intel/AMD/Nvidia GPU.** in our [GitHub repo](https://github.com/xinntao/Real-ESRGAN). <br>
This executable file is **portable** and includes all the binaries and models required. No CUDA or PyTorch environment is needed.<br>
This executable file is based on the wonderful [Tencent/ncnn](https://github.com/Tencent/ncnn).

# 1. Preparations
Before start, make sure that you choose
* Runtime Type = Python 3
* Hardware Accelerator = GPU

in the **Runtime** menu -> **Change runtime type**

Then, we clone the repository, set up the envrironment.

In [ ]:
# Clone Real-ESRGAN and enter the Real-ESRGAN
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
# Set up the environment
!pip install basicsr
!pip install facexlib
!pip install gfpgan
!pip install -r requirements.txt
!python setup.py develop

In [ ]:
print('Uninstalling basicsr, facexlib, and gfpgan to prepare for torchvision upgrade...')
!pip uninstall -y basicsr facexlib gfpgan

In [ ]:
print('Upgrading torch and torchvision to ensure compatibility...')
!pip install torch torchvision --upgrade --index-url https://download.pytorch.org/whl/cu121

In [4]:
print('Reinstalling basicsr, facexlib, gfpgan, and other requirements...')
!pip install basicsr
!pip install facexlib
!pip install gfpgan
!pip install -r requirements.txt
!python setup.py develop

Reinstalling basicsr, facexlib, gfpgan, and other requirements...
  Using cached basicsr-1.4.2-py3-none-any.whl
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
realesrgan 0.3.0 requires facexlib>=0.2.5, which is not installed.
realesrgan 0.3.0 requires gfpgan>=1.3.5, which is not installed.
  Using cached facexlib-0.3.0-py3-none-any.whl.metadata (4.6 kB)
Using cached facexlib-0.3.0-py3-none-any.whl (59 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
realesrgan 0.3.0 requires gfpgan>=1.3.5, which is not installed.
  Using cached gfpgan-1.3.8-py3-none-any.whl.metadata (12 kB)
Using cached gfpgan-1.3.8-py3-none-any.whl (52 kB)
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer 

# 2. Upload Images

Upload the images to be processed by Real-ESRGAN

In [5]:
import os
from google.colab import files
import shutil

upload_folder = 'upload'
result_folder = 'results'

if os.path.isdir(upload_folder):
    shutil.rmtree(upload_folder)
if os.path.isdir(result_folder):
    shutil.rmtree(result_folder)
os.mkdir(upload_folder)
os.mkdir(result_folder)

# upload images
uploaded = files.upload()
for filename in uploaded.keys():
  dst_path = os.path.join(upload_folder, filename)
  print(f'move {filename} to {dst_path}')
  shutil.move(filename, dst_path)

Saving Screenshot 2026-05-03 at 19.07.50.png to Screenshot 2026-05-03 at 19.07.50.png
move Screenshot 2026-05-03 at 19.07.50.png to upload/Screenshot 2026-05-03 at 19.07.50.png


# 3. Inference


In [10]:
# Cell 1 - patch FIRST
import types, sys
import torchvision.transforms.functional as F
fake = types.ModuleType("torchvision.transforms.functional_tensor")
fake.rgb_to_grayscale = F.rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = fake

In [ ]:
# Cell 2 - now safe to install and import
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
!pip install -r requirements.txt -q
!python setup.py develop -q

In [14]:
# Directly patch the degradations.py file on disk
import site

site_packages = site.getsitepackages()[0]
file_path = f"{site_packages}/basicsr/data/degradations.py"

with open(file_path, "r") as f:
    content = f.read()

old_line = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
new_line = "from torchvision.transforms.functional import rgb_to_grayscale"

if old_line in content:
    content = content.replace(old_line, new_line)
    with open(file_path, "w") as f:
        f.write(content)
    print("✅ Patched successfully!")
else:
    print("⚠️ Line not found — may already be patched or path is different.")
    print("File path used:", file_path)

✅ Patched successfully!


In [18]:
# Cell 3 - run inference normally
!python inference_realesrgan.py -i inputs -o results

Testing 0 00003
Testing 1 00017_gray
Testing 2 0014
Testing 3 0030
Testing 4 ADE_val_00000114
Testing 5 OST_009
Testing 6 children-alpha
Testing 7 tree_alpha_16bit
	Input is a 16-bit image
Testing 8 video
⚠️  Skipping inputs/video — could not be read by OpenCV
Testing 9 wolf_gray


In [ ]:
!ls -la /content/Real-ESRGAN/Real-ESRGAN/Real-ESRGAN/inputs/

In [ ]:
# Add this cell to patch the script to skip unreadable files
file_path = "/content/Real-ESRGAN/Real-ESRGAN/Real-ESRGAN/inference_realesrgan.py"

with open(file_path, "r") as f:
    content = f.read()

old = "        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)\n        if len(img.shape)"
new = """        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            print(f'⚠️  Skipping {path} — could not be read by OpenCV')
            continue
        if len(img.shape)"""

content = content.replace(old, new)
with open(file_path, "w") as f:
    f.write(content)
print("✅ Patched!")

In [19]:
# if it is out of memory, try to use the `--tile` option
# We upsample the image with the scale factor X3.5
!python inference_realesrgan.py -n RealESRGAN_x4plus -i upload --outscale 3.5 --face_enhance
# Arguments
# -n, --model_name: Model names
# -i, --input: input folder or image
# --outscale: Output scale, can be arbitrary scale factore.

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth" to /content/Real-ESRGAN/Real-ESRGAN/Real-ESRGAN/gfpgan/weights/detection_Resnet50_Final.pth

100% 104M/104M [00:00<00:00, 370MB/s] 
Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth" to /content/Real-ESRGAN/Real-ESRGAN/Real-ESRGAN/gfpgan/weights/parsing_parsenet.pth

100% 81.4M/81.4M [00:00<00:00, 248MB/s]
Downloading: "https://github.com

# 4. Visualization

In [21]:
# utils for visualization
import cv2
import matplotlib.pyplot as plt
def display(img1, img2):
  fig = plt.figure(figsize=(25, 10))
  ax1 = fig.add_subplot(1, 2, 1)
  plt.title('Input image', fontsize=16)
  ax1.axis('off')
  ax2 = fig.add_subplot(1, 2, 2)
  plt.title('Real-ESRGAN output', fontsize=16)
  ax2.axis('off')
  ax1.imshow(img1)
  ax2.imshow(img2)
def imread(img_path):
  img = cv2.imread(img_path)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  return img

# display each image in the upload folder
import os
import glob

input_folder = 'upload'
result_folder = 'results'
input_list = sorted(glob.glob(os.path.join(input_folder, '*')))
output_list = sorted(glob.glob(os.path.join(result_folder, '*')))
for input_path, output_path in zip(input_list, output_list):
  img_input = imread(input_path)
  img_output = imread(output_path)
  display(img_input, img_output)

# 5. Download Results


In [22]:
# Download the results
zip_filename = 'Real-ESRGAN_result.zip'
if os.path.exists(zip_filename):
  os.remove(zip_filename)
os.system(f"zip -r -j {zip_filename} results/*")
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>